In [ ]:
from typing import List, Optional, Union
from pydantic import BaseModel, Field, ConfigDict


In [ ]:

# --- 1. Global Configuration ---
class ProjectAPIBase(BaseModel):
    # This prevents any fields not defined in the class from being loaded
    model_config = ConfigDict(extra='forbid') 


In [ ]:

# 2. Base Configuration to allow extra fields and nulls
class ProjectAPIBase(BaseModel):
    model_config = ConfigDict(extra='allow') # Permits 'applications' and other extra fields    


In [ ]:

# --- 2. Application Models (New) ---
# Your project file includes an 'applications' array not in the main schema
class Application(ProjectAPIBase):
    uuid: str
    type: str
    device_template_uri: str
    device_template_id: str
    device_manifest_uri: Optional[str] = None
    image_uri: Optional[str] = None
    module_id: Optional[str] = None
    binary_uri: Optional[str] = None
    binary_id: Optional[int] = None
    protocol: Optional[int] = None
    bluest_v3_payload: Optional[dict] = None

# --- 3. Deployment Sub-Models ---
class LeafInferenceModel(ProjectAPIBase):
    model_name_reference: str
    artifact_type: str
    component_name: str

class LeafDevice(ProjectAPIBase):
    device_id: str
    display_name: str
    gateway_id: Optional[str]
    description: str
    datalogging: dict
    inference: dict

class GatewayDevice(ProjectAPIBase):
    device_id: str
    display_name: str
    gateway_id: Optional[Union[str, None]]
    description: str
    application: str
    wifi_mode: str

class Deployment(ProjectAPIBase):
    uuid: str
    display_name: str
    description: str
    last_update_time: Optional[Union[str, None]]
    last_deploy_result: Optional[Union[str, None]]
    cloud_params: dict
    gateway: List[GatewayDevice] # Moved from schema-nested to project-flat
    leaf: List[LeafDevice]

# --- 4. Model Sub-Models ---
class AIModel(ProjectAPIBase):
    uuid: str
    name: str
    description: Optional[str] = None
    creation_time: Optional[Union[str, None]] # Handles 'null'
    last_update_time: Optional[Union[str, None]]
    model_owner_uuid: Optional[Union[str, None]]
    metadata: dict
    dataset: dict
    target: dict
    data_sufficiency: Optional[dict] = None # Field found in project but not main schema
    training: Optional[dict] = None
    codegen: Optional[Union[dict, None]]
    building: Optional[Union[dict, None]]

# --- 5. Main Project Model ---
class AIProject(ProjectAPIBase):
    # Required by JSON Schema
    dollar_schema: str = Field(alias="$schema") 
    schema_version: str # Field from project file
    ai_project_name: str
    display_name: str
    description: str
    uuid: str
    version: str = Field(..., pattern=r"^[0-9].[0-9].[0-9]$")
    creation_time: Optional[Union[str, None]]
    last_update_time: Optional[Union[str, None]]
    project_owner_uuid: Optional[Union[str, None]]
    models: List[AIModel]
    applications: List[Application] # Explicitly handling the additional section
    deployments: List[Deployment]

    type: str  # Matches "industrial" in your JSON
    project_owner_uuid: Optional[Union[str, None]] # Matches the null in your JSON    

In [ ]:
import json
from pathlib import Path
from pydantic import ValidationError
from pprint import pprint


# 1. Define the path to your file
file_path = Path("ai_get_started_smart_asset_tracking_md_mlc.json")

# 2. Open and parse the raw JSON into a Python dictionary
try:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    
    # 3. Instantiate the Pydantic model
    # The **raw_data unpacks the dictionary keys as arguments to the class
    project = AIProject(**raw_data)

    # 4. Access data with dot notation (fully type-hinted in VS Code)
    print(f"Successfully loaded: {project.display_name} (v{project.version})")
    print(f"Project UUID: {project.uuid}")
    
    # Accessing nested models
    for model in project.models:
        print(f"Found Model: {model.name}")
        print(f" - Target Hardware: {model.target.get('device')}")

    pprint(project.model_dump())  # Print the entire project as a dictionary

except FileNotFoundError:
    print(f"Error: The file {file_path} was not found.")
except json.JSONDecodeError:
    print("Error: The file is not a valid JSON format.")
except ValidationError as e:
    # This will trigger if the JSON violates your 'ProjectAPIBase' rules
    print("Validation Error: The JSON structure does not match the Pydantic model.")
    print(e.json(indent=2))

Successfully loaded: Smart Asset Tracking MD (v1.0.0)
Project UUID: uuid_to_replace_project
Found Model: smart_asset_tracking
 - Target Hardware: steval-mkboxpro
{'ai_project_name': 'get_started_smart_asset_tracking_md_mlc',
 'applications': [{'binary_id': None,
                   'binary_uri': None,
                   'bluest_v3_payload': None,
                   'device_manifest_uri': '/e2e_systems/gateway/dtmi/appconfig/gateway/MLC_Gateway-8.1.manifest.arm64v8.json',
                   'device_template_id': 'dtmi:appconfig:gateway:MLC_Gateway;8',
                   'device_template_uri': '/e2e_systems/gateway/dtmi/appconfig/gateway/MLC_Gateway-8.expanded.json',
                   'image_uri': '/e2e_systems/gateway/images/raspberrypi/staiotcraft_rpi_gateway_1_5_0.img.gz',
                   'module_id': 'edgeMLC',
                   'protocol': None,
                   'type': 'gateway',
                   'uuid': 'uuid_to_replace_application1'},
                  {'binary_id': 381,
